## Import libraries

In [ ]:
import gc
import json
import logging
import math
import os
import random
import re
import time
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer



## Environment helpers

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def mixed_precision_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda":
        return torch.cuda.amp.autocast()
    return nullcontext()


def resolve_existing_path(*candidates: str) -> str:
    for cand in candidates:
        if not cand:
            continue
        p = Path(cand)
        if p.is_file() and p.name == "config.json":
            return str(p.parent)
        if p.is_dir():
            if (p / "config.json").exists():
                return str(p)
            nested = sorted(p.rglob("config.json"))
            if nested:
                return str(nested[0].parent)
    raise FileNotFoundError(f"None of the candidate paths exist: {candidates}")


def resolve_test_data_path(path: str) -> str:
    if os.path.exists(path):
        return path
    alternates = [
        "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv",
        "/kaggle/input/deep-past-initiative-machine-translation/test.csv",
    ]
    for alt in alternates:
        if os.path.exists(alt):
            return alt
    raise FileNotFoundError("Could not find test.csv. Attach the competition dataset and set cfg.test_data_path accordingly.")


def setup_logging(output_dir: str) -> logging.Logger:
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir) / "three_model_ensemble.log"),
        ],
    )
    return logging.getLogger("three_model_ensemble")



## Configuration and model setup

In [ ]:
@dataclass(frozen=True)
class ModelRunSpec:
    label: str
    model_path: str
    preprocessor_kind: str
    input_prefix: str
    num_beams: int = 4
    length_penalty: float = 1.3
    repetition_penalty: Optional[float] = None
    max_new_tokens: int = 256
    tokenizer_path: str = ""


@dataclass
class EnsembleConfig:
    test_data_path: str = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
    output_dir: str = "/kaggle/working/"

    base_model_path: str = "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x"
    mbr_v2_model_path: str = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1"
    mbr6_model_path: str = "/kaggle/input/mattiaangelibyt5-akkadian-mbrpytorchdefault6"

    max_input_length: int = 512
    batch_size: int = 6
    num_workers: int = 2
    num_buckets: int = 24

    use_mixed_precision: bool = True
    use_better_transformer: bool = True
    use_bucket_batching: bool = True
    use_adaptive_beams: bool = True
    checkpoint_freq: int = 2000
    empty_cache_every: int = 12
    validate_samples: int = 6
    seed: int = 42

    model_specs: Tuple[ModelRunSpec, ...] = field(default_factory=tuple)

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(parents=True, exist_ok=True)
        if self.device.type == "cpu":
            self.use_mixed_precision = False
            self.use_better_transformer = False
            self.batch_size = min(self.batch_size, 2)

        if not self.model_specs:
            self.model_specs = (
                ModelRunSpec(
                    label="base_len115",
                    model_path=resolve_existing_path(
                        self.base_model_path,
                        "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x",
                    ),
                    preprocessor_kind="base",
                    input_prefix="translate Akkadian to English: ",
                    num_beams=6,
                    length_penalty=1.15,
                    repetition_penalty=None,
                    max_new_tokens=256,
                ),
                ModelRunSpec(
                    label="mbr_v2",
                    model_path=resolve_existing_path(
                        self.mbr_v2_model_path,
                        "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
                        "/kaggle/input/byt5-akkadian-mbr-v2/PyTorch/default/1",
                        "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
                    ),
                    preprocessor_kind="mbr_v2",
                    input_prefix="",
                    num_beams=4,
                    length_penalty=1.3,
                    repetition_penalty=None,
                    max_new_tokens=256,
                ),
                ModelRunSpec(
                    label="mbr6",
                    model_path=resolve_existing_path(
                        self.mbr6_model_path,
                        "/kaggle/input/datasets/spencevanasperdt/mattiaangelibyt5-akkadian-mbrpytorchdefault6",
                        "/kaggle/input/spencevanasperdt-mattiaangelibyt5-akkadian-mbrpytorchdefault6",
                        "/kaggle/input/mattiaangelibyt5-akkadian-mbrpytorchdefault6/model",
                    ),
                    preprocessor_kind="mbr6",
                    input_prefix="translate Akkadian to English: ",
                    num_beams=4,
                    length_penalty=1.25,
                    repetition_penalty=None,
                    max_new_tokens=256,
                ),
            )



## Preprocessing

In [ ]:
V2_RE = re.compile(r"([aAeEiIuU])(?:2|\u2082)")
V3_RE = re.compile(r"([aAeEiIuU])(?:3|\u2083)")
ACUTE = str.maketrans({"a": "a", "e": "e", "i": "i", "u": "u", "A": "A", "E": "E", "I": "I", "U": "U"})
GRAVE = str.maketrans({"a": "a", "e": "e", "i": "i", "u": "u", "A": "A", "E": "E", "I": "I", "U": "U"})


def ascii_to_diacritics(text: str) -> str:
    s = str(text or "")
    s = s.replace("sz", "sh").replace("SZ", "Sh")
    s = s.replace("s,", "s").replace("S,", "S")
    s = s.replace("t,", "t").replace("T,", "T")
    s = V2_RE.sub(lambda m: m.group(1).translate(ACUTE), s)
    s = V3_RE.sub(lambda m: m.group(1).translate(GRAVE), s)
    return s


ALLOWED_FRACS = [
    (1 / 6, "0.16666"),
    (1 / 4, "0.25"),
    (1 / 3, "0.33333"),
    (1 / 2, "0.5"),
    (2 / 3, "0.66666"),
    (3 / 4, "0.75"),
    (5 / 6, "0.83333"),
]
FRAC_TOL = 2e-3
FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
WS_RE = re.compile(r"\s+")


def canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|\u2026+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I,
)

CHAR_TRANS = str.maketrans({
    "\u1e2b": "h",
    "\u1e2a": "H",
    "\u02be": "",
    "\u2080": "0",
    "\u2081": "1",
    "\u2082": "2",
    "\u2083": "3",
    "\u2084": "4",
    "\u2085": "5",
    "\u2086": "6",
    "\u2087": "7",
    "\u2088": "8",
    "\u2089": "9",
    "\u2014": "-",
    "\u2013": "-",
})
SUB_X = "\u2093"

DET_UPPER_RE = re.compile(r"\(([A-Z0-9]{1,6})\)")
DET_LOWER_RE = re.compile(r"\(([a-z]{1,4})\)")
PN_RE = re.compile(r"\bPN\b")
SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)(?:\.\s*(?:plur|plural|sing|singular))?\.?\s*[^)]*\)",
    re.I,
)
BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)
REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")
FORBIDDEN_TRANS = str.maketrans("", "", "()<>[]+;")
MULTI_GAP_RE = re.compile(r"(?:<gap>\s*){2,}")


class Preprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts, dtype="object").fillna("").astype(str)
        ser = ser.apply(ascii_to_diacritics)
        ser = ser.str.replace(DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(DET_LOWER_RE, r"{\1}", regex=True)
        ser = ser.str.replace(GAP_UNIFIED_RE, "<gap>", regex=True)
        ser = ser.str.replace(r"(?:<gap>\s*){2,}", "<big_gap> ", regex=True)
        ser = ser.str.translate(CHAR_TRANS)
        ser = ser.str.replace(SUB_X, "", regex=False)
        ser = ser.str.replace(FLOAT_RE, lambda m: canon_decimal(float(m.group(1))), regex=True)
        ser = ser.str.replace(WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


_MODEL_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_MODEL_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_MODEL_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_MODEL_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})


def _ascii_to_diacritics_v2(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _MODEL_V2.sub(lambda m: m.group(1).translate(_MODEL_ACUTE), s)
    s = _MODEL_V3.sub(lambda m: m.group(1).translate(_MODEL_GRAVE), s)
    return s


_ALLOWED_FRACS_V2 = [
    (1/6, "0.16666"), (1/4, "0.25"), (1/3, "0.33333"),
    (1/2, "0.5"), (2/3, "0.66666"), (3/4, "0.75"), (5/6, "0.83333"),
]
_FRAC_TOL_V2 = 2e-3
_FLOAT_RE_V2 = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")


def _canon_decimal_v2(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS_V2, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= _FRAC_TOL_V2:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


_GAP_UNIFIED_RE_V2 = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I,
)


def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE_V2, "<gap>", regex=True)


_CHAR_TRANS_V2 = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X_V2 = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE_V2 = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE_V2 = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")
_KUBABBAR_RE_V2 = re.compile(r"KÙ\.B\.")
_EXACT_FRAC_RE_V2 = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP_V2 = {
    "0.8333": "⅚", "0.6666": "⅔", "0.3333": "⅓", "0.1666": "⅙",
    "0.625": "⅝", "0.75": "¾", "0.25": "¼", "0.5": "½",
}
_WS_RE_V2 = re.compile(r"\s+")


def _frac_repl_v2(m: re.Match) -> str:
    return _EXACT_FRAC_MAP_V2[m.group(0)]


class MattiaV2Preprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics_v2)
        ser = ser.str.replace(_DET_UPPER_RE_V2, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE_V2, r"{\1}", regex=True)
        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS_V2)
        ser = ser.str.replace(_SUB_X_V2, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE_V2, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_EXACT_FRAC_RE_V2, _frac_repl_v2, regex=True)
        ser = ser.str.replace(_FLOAT_RE_V2, lambda m: _canon_decimal_v2(float(m.group(1))), regex=True)
        ser = ser.str.replace(_WS_RE_V2, " ", regex=True).str.strip()
        return ser.tolist()


OA_HINTS = {
    "KÙ.BABBAR":"silver","KU.BABBAR":"silver","AN.NA":"tin","MA.NA":"mina",
    "GÍN":"shekel","GIN":"shekel","TÚG":"textile","TUG":"textile",
    "GADA":"linen","ŠE":"barley","SE":"barley","URUDU":"copper",
    "ANŠE":"donkey","É":"house","DUMU":"son","DUMU.MUNUS":"daughter",
}

_GAP_RE_MBR6 = re.compile(
    r"<\s*big[\s_\-]*gap\s*>|<\s*gap\s*>|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b|\.{3,}|…+|\[\.+\]|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)|\(\s*break\s*\)|\(\s*\d+\s+broken\s+lines?\s*\)", re.I
)
_CHAR_TRANS_MBR6 = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"","₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9","—":"-","–":"-",
})
_WS_RE_MBR6 = re.compile(r"\s+")
_FLOAT_RE_MBR6 = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
_ALLOWED_FRACS_MBR6 = [
    (1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),(1/2,"0.5"),
    (2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333"),
]


def canon_decimal_mbr6(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS_MBR6, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= 2e-3:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


class MBR6Preprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        results = []
        for text in texts:
            t = _ascii_to_diacritics_v2(str(text))
            t = _GAP_RE_MBR6.sub("<gap>", t)
            t = t.translate(_CHAR_TRANS_MBR6).replace("ₓ", "")
            t = _FLOAT_RE_MBR6.sub(lambda m: canon_decimal_mbr6(float(m.group(1))), t)
            t = _WS_RE_MBR6.sub(" ", t).strip()
            found = [v for k, v in OA_HINTS.items() if k.upper() in t.upper()]
            if found:
                t += f" [Commodities: {', '.join(list(dict.fromkeys(found))[:4])}]"
            results.append(t)
        return results


def build_preprocessor(kind: str):
    kind = (kind or "base").lower()
    if kind == "base":
        return Preprocessor()
    if kind in {"mbr_v2", "optimized_v2"}:
        return MattiaV2Preprocessor()
    if kind in {"mbr6", "oa_hints"}:
        return MBR6Preprocessor()
    raise ValueError(f"Unsupported preprocessor kind: {kind}")



## Postprocessing and heuristic fusion

In [ ]:
class SafePostprocessor:
    SUBSCRIPT_TO_NORMAL = str.maketrans({
        "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4","₅":"5","₆":"6","₇":"7","₈":"8","₉":"9"
    })

    def __init__(self, use_unicode_fractions: bool = False, strip_dash_old: bool = False):
        self.use_unicode_fractions = use_unicode_fractions
        self.strip_dash_old = strip_dash_old
        self.forbidden_chars = re.compile(r"[⌈⌋⌊⌉]")
        self.multi_space = re.compile(r"\s+")
        self.space_before_punct = re.compile(r"\s+([,.;:!?])")
        self.multi_punct = re.compile(r"([!?.,])\1{2,}")
        self.dashes = re.compile(r"[–—]")
        self.prot_gap = "\uFFF0"
        self.prot_big = "\uFFF1"
        self.bracket_x = re.compile(r"\[\s*x\s*\]|\(\s*x\s*\)", re.IGNORECASE)
        self.bare_x = re.compile(r"(?:(?<=\s)|^)x(?=(\s|$))", re.IGNORECASE)
        self.ellipsis = re.compile(r"(\.\.\.+|…+)")
        self.frac_map = {"0.5": "½", "0.25": "¼", "0.75": "¾", "1/2": "½", "1/4": "¼", "3/4": "¾"}
        self.frac_re = re.compile(r"\b(0\.5|0\.25|0\.75|1/2|1/4|3/4)\b")

    def postprocess_one(self, text: str) -> str:
        t = str(text or "")
        t = t.replace("ḫ", "h").replace("Ḫ", "H")
        t = t.translate(self.SUBSCRIPT_TO_NORMAL)
        t = self.dashes.sub("-", t)
        t = self.bracket_x.sub("<gap>", t)
        t = self.bare_x.sub("<gap>", t)
        t = self.ellipsis.sub("<big_gap>", t)
        t = t.replace("<gap>", self.prot_gap).replace("<big_gap>", self.prot_big)
        t = self.forbidden_chars.sub("", t)
        t = t.replace(self.prot_gap, "<gap>").replace(self.prot_big, "<big_gap>")
        if self.strip_dash_old:
            t = t.strip(" -")
        if self.use_unicode_fractions:
            t = self.frac_re.sub(lambda m: self.frac_map.get(m.group(1), m.group(1)), t)
        t = self.space_before_punct.sub(r"\1", t)
        t = self.multi_punct.sub(r"\1", t)
        t = self.multi_space.sub(" ", t).strip()
        return t

    def postprocess_batch(self, texts: List[str]) -> List[str]:
        return [self.postprocess_one(x) for x in texts]


class AggressivePostprocessor:
    def __init__(self, ngram_dedupe_max_n: int = 4):
        self.ngram_dedupe_max_n = ngram_dedupe_max_n
        self.pat_gap = re.compile(r"(\[x\]|\(x\)|\bx\b)", re.I)
        self.pat_big_gap = re.compile(r"(\.{3,}|…|\[\.+\])")
        self.pat_gap_runs = re.compile(r"(?:<gap>\s*){2,}")
        self.pat_biggap_runs = re.compile(r"(?:<big_gap>\s*){2,}")
        self.pat_annotations = re.compile(r"\((fem|plur|pl|sing|singular|plural|masc|uncertain|\?|!|damaged|broken)\.?\s*\w*\)", re.I)
        self.pat_repeat_word = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.I)
        self.pat_punct_space = re.compile(r"\s+([.,:;!?])")
        self.pat_multi_punct = re.compile(r"([.,!?])\1+")
        self.pat_whitespace = re.compile(r"\s+")
        self.subscript_trans = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
        self.special_chars_trans = str.maketrans("ḫḪ", "hH")
        self.forbidden_chars = '!?()"—–<>⌈⌋⌊⌉[]+ʾ/;'
        self.forbidden_trans = str.maketrans("", "", self.forbidden_chars)

    def _dedupe_ngrams(self, text: str) -> str:
        tokens = text.split()
        if len(tokens) < 5:
            return text
        for n in range(self.ngram_dedupe_max_n, 1, -1):
            i = 0
            while i <= len(tokens) - 2 * n:
                if tokens[i:i+n] == tokens[i+n:i+2*n]:
                    del tokens[i+n:i+2*n]
                else:
                    i += 1
        return " ".join(tokens)

    def postprocess_one(self, text: str) -> str:
        if not isinstance(text, str) or not text.strip():
            return ""
        t = text.translate(self.special_chars_trans).translate(self.subscript_trans)
        t = self.pat_gap.sub("<gap>", t)
        t = self.pat_big_gap.sub("<big_gap>", t)
        t = self.pat_gap_runs.sub("<big_gap> ", t)
        t = self.pat_biggap_runs.sub("<big_gap> ", t)
        t = self.pat_annotations.sub("", t)
        t = t.replace("<gap>", "\x00GAP\x00").replace("<big_gap>", "\x00BIG\x00")
        t = t.translate(self.forbidden_trans)
        t = t.replace("\x00GAP\x00", " <gap> ").replace("\x00BIG\x00", " <big_gap> ")
        t = self.pat_repeat_word.sub(r"\1", t)
        t = self._dedupe_ngrams(t)
        t = self.pat_punct_space.sub(r"\1", t)
        t = self.pat_multi_punct.sub(r"\1", t)
        t = self.pat_whitespace.sub(" ", t)
        return t.strip().strip("-").strip()

    def postprocess_batch(self, translations: List[str]) -> List[str]:
        return [self.postprocess_one(t) for t in translations]


def badness_score(text: str) -> float:
    t = str(text or "").strip()
    if not t:
        return 10.0
    words = t.split()
    n = len(words)
    score = 0.0
    if n < 5:
        score += 3.0
    if n < 3:
        score += 3.0
    if n > 500:
        score += 2.0
    gaps = t.count("<gap>") + t.count("<big_gap>")
    if gaps > 6:
        score += (gaps - 6) * 0.35
    rep = 0
    for i in range(2, n):
        if words[i].lower() == words[i-1].lower() == words[i-2].lower():
            rep += 1
    score += rep * 0.75
    return score


KEYWORDS = {
    "tablet", "king", "city", "god", "silver", "gold", "temple", "house", "palace",
    "year", "son", "daughter", "brother", "mother", "father", "gave", "took", "sent",
    "received", "grain", "sheep", "oil", "wine"
}


def quality_score(text: str) -> float:
    t = str(text or "").strip()
    if not t:
        return 0.0
    words = t.split()
    n = len(words)
    score = 0.0
    if 50 <= n <= 200:
        score += 1.0
    if t[0].isupper():
        score += 0.5
    if t.endswith((".", "?", "!")):
        score += 0.5
    lw = {w.strip(".,;:!?\"'()[]").lower() for w in words}
    score += min(2.0, 0.25 * len(lw.intersection(KEYWORDS)))
    gaps = t.count("<gap>") + t.count("<big_gap>")
    score -= min(2.0, 0.15 * gaps)
    score -= 0.5 * max(0.0, badness_score(t) - 1.0)
    return score


def fuse_texts(a: str, b: str, tie_badness_thresh=0.5, w_a=0.6, w_b=0.4, prefer="a") -> str:
    ba = badness_score(a)
    bb = badness_score(b)
    qa = quality_score(a) * w_a
    qb = quality_score(b) * w_b
    if ba + tie_badness_thresh < bb:
        return a
    if bb + tie_badness_thresh < ba:
        return b
    if qa > qb:
        return a
    if qb > qa:
        return b
    return a if prefer == "a" else b


def finalize_translation(text: str, min_words: int = 3) -> str:
    tt = str(text or "").strip()
    if not tt:
        return "The tablet contains fragmentary text."
    if len(tt.split()) < min_words:
        return "The tablet contains an incomplete inscription."
    if tt[0].isalpha() and tt[0].islower():
        tt = tt[0].upper() + tt[1:]
    if not tt.endswith((".", "!", "?")):
        tt += "."
    tt = re.sub(r"\s+([,.;:!?])", r"\1", tt)
    tt = re.sub(r"\s+", " ", tt).strip()
    return tt



## Dataset and bucketed batching

In [ ]:
class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor, input_prefix: str):
        self.ids = df["id"].astype(str).tolist()
        raw = df["transliteration"].astype("object").fillna("").tolist()
        proc = preprocessor.preprocess_batch(raw)
        self.inputs = [f"{input_prefix}{x}" for x in proc]

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        return self.ids[idx], self.inputs[idx]


class BucketSampler(Sampler[List[int]]):
    def __init__(self, texts: List[str], batch_size: int, num_buckets: int = 24, shuffle: bool = False):
        self.batch_size = batch_size
        lengths = np.array([max(1, len(t.split())) for t in texts], dtype=np.int32)
        order = np.argsort(lengths)
        self.indices = order.tolist()
        self.num_buckets = max(1, num_buckets)
        self.buckets = np.array_split(self.indices, self.num_buckets)
        self.batches = []
        rng = np.random.default_rng(42)
        for bucket in self.buckets:
            bucket = list(bucket)
            if shuffle:
                rng.shuffle(bucket)
            for i in range(0, len(bucket), batch_size):
                chunk = bucket[i:i + batch_size]
                if chunk:
                    self.batches.append(chunk)
        if shuffle:
            rng.shuffle(self.batches)

    def __iter__(self):
        for batch in self.batches:
            yield batch

    def __len__(self):
        return len(self.batches)



## Model loading and single-pass generation

In [ ]:
class ModelWrapper:
    def __init__(self, spec: ModelRunSpec, cfg: EnsembleConfig, logger: logging.Logger):
        self.spec = spec
        self.cfg = cfg
        self.logger = logger
        self.device = cfg.device
        tokenizer_path = spec.tokenizer_path or spec.model_path
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, local_files_only=True)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(spec.model_path, local_files_only=True).to(self.device)
        self.model.eval()
        self.logger.info(f"[{spec.label}] loaded: {spec.model_path}")
        if cfg.use_better_transformer and self.device.type == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                self.logger.info(f"[{spec.label}] BetterTransformer enabled")
            except Exception as e:
                self.logger.info(f"[{spec.label}] BetterTransformer skipped: {e}")

    def collate(self, batch):
        ids, texts = zip(*batch)
        enc = self.tokenizer(
            list(texts),
            max_length=self.cfg.max_input_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return list(ids), enc

    def adaptive_beams(self, attention_mask: torch.Tensor) -> int:
        if not self.cfg.use_adaptive_beams:
            return self.spec.num_beams
        lens = attention_mask.sum(dim=1).detach().cpu().numpy()
        med = float(np.median(lens)) if len(lens) else 0.0
        if med < 100:
            return max(2, self.spec.num_beams // 2)
        return self.spec.num_beams

    def generate_batch(self, enc) -> List[str]:
        input_ids = enc["input_ids"].to(self.device, non_blocking=True)
        attn = enc["attention_mask"].to(self.device, non_blocking=True)
        beams = self.adaptive_beams(attn)
        gen_kwargs = dict(
            num_beams=beams,
            max_new_tokens=self.spec.max_new_tokens,
            length_penalty=self.spec.length_penalty,
            early_stopping=True,
            no_repeat_ngram_size=0,
        )
        if self.spec.repetition_penalty is not None:
            gen_kwargs["repetition_penalty"] = self.spec.repetition_penalty

        with torch.inference_mode():
            with mixed_precision_ctx(self.device, self.cfg.use_mixed_precision):
                out_ids = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attn,
                    **gen_kwargs,
                )
        return self.tokenizer.batch_decode(out_ids, skip_special_tokens=True)

    def unload(self):
        try:
            from optimum.bettertransformer import BetterTransformer
            self.model = BetterTransformer.reverse(self.model)
        except Exception:
            pass
        del self.model
        del self.tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        self.logger.info(f"[{self.spec.label}] unloaded")



## End-to-end three-model ensemble engine

In [ ]:
class ThreeModelEnsembleEngine:
    def __init__(self, cfg: EnsembleConfig, logger: logging.Logger):
        self.cfg = cfg
        self.logger = logger
        self.safe_pp = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=False)
        self.safe_pp_stripdash = SafePostprocessor(use_unicode_fractions=False, strip_dash_old=True)
        self.agg_pp = AggressivePostprocessor()

    def postprocess_batch(self, texts: List[str]) -> List[str]:
        safe = self.safe_pp.postprocess_batch(texts)
        refined = []
        for text in safe:
            refined.append(self.agg_pp.postprocess_one(text) if badness_score(text) >= 1.5 else text)
        return [finalize_translation(x) for x in refined]

    def run_one_model(self, spec: ModelRunSpec, test_df: pd.DataFrame) -> pd.DataFrame:
        dataset = AkkadianDataset(test_df, build_preprocessor(spec.preprocessor_kind), spec.input_prefix)
        wrapper = ModelWrapper(spec, self.cfg, self.logger)

        if self.cfg.use_bucket_batching:
            sampler = BucketSampler(dataset.inputs, batch_size=self.cfg.batch_size, num_buckets=self.cfg.num_buckets)
            dl = DataLoader(dataset, batch_sampler=sampler, num_workers=self.cfg.num_workers, pin_memory=(self.cfg.device.type == "cuda"), collate_fn=wrapper.collate)
        else:
            dl = DataLoader(dataset, batch_size=self.cfg.batch_size, shuffle=False, num_workers=self.cfg.num_workers, pin_memory=(self.cfg.device.type == "cuda"), collate_fn=wrapper.collate)

        rows: List[Tuple[str, str]] = []
        t0 = time.time()
        for step, (ids, enc) in enumerate(dl):
            decoded = wrapper.generate_batch(enc)
            cleaned = self.postprocess_batch(decoded)
            rows.extend(zip(ids, cleaned))

            if self.cfg.checkpoint_freq and len(rows) % self.cfg.checkpoint_freq == 0:
                ck = pd.DataFrame(rows, columns=["id", "translation"])
                ck_path = Path(self.cfg.output_dir) / f"checkpoint_{spec.label}_{len(rows)}.csv"
                ck.to_csv(ck_path, index=False)
                self.logger.info(f"saved checkpoint: {ck_path}")

            if self.cfg.device.type == "cuda" and self.cfg.empty_cache_every and (step + 1) % self.cfg.empty_cache_every == 0:
                torch.cuda.empty_cache()

            if (step + 1) % 50 == 0:
                self.logger.info(f"[{spec.label}] step={step+1}/{len(dl)} done={len(rows)} elapsed={time.time()-t0:.1f}s")

        df = pd.DataFrame(rows, columns=["id", "translation"])
        wrapper.unload()
        del wrapper
        self.validate(df, spec.label)
        out_path = Path(self.cfg.output_dir) / f"submission_{spec.label}.csv"
        df.to_csv(out_path, index=False)
        self.logger.info(f"saved model submission: {out_path}")
        return df

    def fuse_pair(self, df_a: pd.DataFrame, df_b: pd.DataFrame, prefer: str = "a", w_a: float = 0.6, w_b: float = 0.4) -> pd.DataFrame:
        a = df_a.set_index("id")["translation"]
        b = df_b.set_index("id")["translation"]
        rows = []
        for _id in a.index:
            fused = fuse_texts(a.loc[_id], b.loc[_id], prefer=prefer, tie_badness_thresh=0.5, w_a=w_a, w_b=w_b)
            rows.append((_id, finalize_translation(fused)))
        return pd.DataFrame(rows, columns=["id", "translation"])

    def run(self, test_df: pd.DataFrame) -> Tuple[Dict[str, pd.DataFrame], pd.DataFrame]:
        submissions: Dict[str, pd.DataFrame] = {}
        for spec in self.cfg.model_specs:
            submissions[spec.label] = self.run_one_model(spec, test_df)

        fused_12 = self.fuse_pair(submissions["base_len115"], submissions["mbr_v2"], prefer="a", w_a=0.58, w_b=0.42)
        fused_12_path = Path(self.cfg.output_dir) / "submission_fused_base_mbrv2.csv"
        fused_12.to_csv(fused_12_path, index=False)
        self.logger.info(f"saved fused submission: {fused_12_path}")

        final_df = self.fuse_pair(fused_12, submissions["mbr6"], prefer="a", w_a=0.55, w_b=0.45)
        final_path = Path(self.cfg.output_dir) / "submission.csv"
        final_df.to_csv(final_path, index=False)
        self.logger.info(f"saved final submission: {final_path}")
        self.validate(final_df, "final")
        return submissions, final_df

    def validate(self, df: pd.DataFrame, tag: str):
        if df.empty:
            self.logger.warning(f"[{tag}] empty output dataframe")
            return
        lens = df["translation"].fillna("").map(lambda x: len(str(x).split()))
        empty_pct = (df["translation"].fillna("").str.strip() == "").mean() * 100
        self.logger.info(
            f"[{tag}] rows={len(df)} empty%={empty_pct:.2f} len(mean/med/min/max)={lens.mean():.1f}/{lens.median():.1f}/{lens.min()}/{lens.max()}"
        )



## Run inference and export submission.csv

In [ ]:
def print_env(cfg: EnsembleConfig):
    print(f"PyTorch : {torch.__version__}")
    print(f"CUDA    : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU     : {torch.cuda.get_device_name(0)}")
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"GPU Mem : {total:.2f} GB")


def main():
    cfg = EnsembleConfig()
    set_seed(cfg.seed)
    logger = setup_logging(cfg.output_dir)
    cfg.test_data_path = resolve_test_data_path(cfg.test_data_path)

    print_env(cfg)
    logger.info(f"Loading test data: {cfg.test_data_path}")
    test_df = pd.read_csv(cfg.test_data_path, encoding="utf-8")
    logger.info(f"Test rows: {len(test_df)}")

    engine = ThreeModelEnsembleEngine(cfg, logger)
    submissions, final_df = engine.run(test_df)

    snapshot = {
        "test_data_path": cfg.test_data_path,
        "model_specs": [asdict(spec) for spec in cfg.model_specs],
        "batch_size": cfg.batch_size,
        "num_buckets": cfg.num_buckets,
        "use_bucket_batching": cfg.use_bucket_batching,
        "submission_files": {label: f"submission_{label}.csv" for label in submissions},
        "final_submission": "submission.csv",
    }
    with open(Path(cfg.output_dir) / "three_model_ensemble_config.json", "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False, indent=2)

    print(final_df.head())
    print(final_df.tail())


if __name__ == "__main__":
    main()

